# 12. PyTorch 입문 — 텐서와 자동미분

> **제12장** · **이론편 대응: 10.4절(역전파), 11.1절(최적화), 11.6절(에폭·배치)**
> **예상 소요**: 60분
> **필요 사양**: CPU로 가능 (GPU 있으면 더 빠름)

---

## 이 장에서 하는 일

11장에서 손으로 만든 역전파를 PyTorch로 다시 한다. **핵심은 값이 같은지 확인하는 것**이다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | 텐서 — NumPy와 무엇이 다른가 | — |
| 2 | **autograd로 이론편 10.4절 값 재현** ★ | 10.4절 |
| 3 | 계산 그래프 들여다보기 | 5.3절 |
| 4 | `nn.Module`로 신경망 만들기 | 10.2절 |
| 5 | 옵티마이저 비교 | 11.1절 |
| 6 | 학습 루프의 표준 형태 | 11.6절 |
| 7 | GPU 사용 |  |

**2절이 이 장의 핵심이다.** PyTorch의 `backward()`가 우리가 손으로 구한 것과
같은 계산을 하고 있다는 사실을 확인한다.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

print("=" * 50)
print(f"PyTorch 버전 : {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
else:
    print("GPU 없음 — 이 장은 CPU로 전부 실행됩니다")
print("=" * 50)

---

## 1. 텐서 — NumPy 배열과 무엇이 다른가

PyTorch의 텐서는 NumPy 배열과 사용법이 거의 같다. 실제로 서로 변환도 된다.

**차이는 두 가지다.**

| 항목 | NumPy | PyTorch |
|---|---|---|
| GPU 사용 | 불가 | 가능 (`.cuda()`) |
| 자동 미분 | 불가 | 가능 (`requires_grad=True`) |

두 번째가 결정적이다. 11장에서 우리는 미분식을 손으로 유도해 코드로 옮겼다.
PyTorch는 그것을 **자동으로 해 준다.**

In [ ]:
import torch
import numpy as np

print("=" * 55)
print("NumPy와 PyTorch — 거의 같은 사용법")
print("=" * 55)

# 생성
a_np = np.array([[1.0, 2.0], [3.0, 4.0]])
a_pt = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

print(f"NumPy   : {a_np.shape}, {a_np.dtype}")
print(f"PyTorch : {tuple(a_pt.shape)}, {a_pt.dtype}")
print()

# 연산도 같다
b_np = np.array([[5.0, 6.0], [7.0, 8.0]])
b_pt = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

print("행렬 곱 (@)")
print(f"  NumPy   :\n{a_np @ b_np}")
print(f"  PyTorch :\n{(a_pt @ b_pt).numpy()}")
print(f"  일치: {np.allclose(a_np @ b_np, (a_pt @ b_pt).numpy())}")
print()

# 상호 변환
converted = torch.from_numpy(a_np)
back = converted.numpy()
print(f"NumPy → Tensor → NumPy : {np.allclose(a_np, back)}")
print()

# 이론편 4.2절 값으로 확인
A = torch.tensor([[2.0, -1.0], [1.0, 3.0]])
x = torch.tensor([3.0, 2.0])
print(f"이론편 4.2절: A @ x = {(A @ x).numpy()}  (기대 [4., 9.])")
assert torch.allclose(A @ x, torch.tensor([4.0, 9.0]))
print("[OK]")

### 자료형에 주의

PyTorch는 NumPy보다 자료형에 엄격하다. NumPy는 기본이 `float64`인데
PyTorch는 `float32`다. 섞으면 오류가 나기도 한다.

In [ ]:
import torch
import numpy as np

print("=" * 55)
print("자료형 차이")
print("=" * 55)
print(f"NumPy 기본   : {np.array([1.0, 2.0]).dtype}")
print(f"PyTorch 기본 : {torch.tensor([1.0, 2.0]).dtype}")
print()

# from_numpy는 원본 자료형을 유지한다
t = torch.from_numpy(np.array([1.0, 2.0]))
print(f"from_numpy 결과: {t.dtype}   ← float64가 그대로 옴")
print()

# 자료형이 다르면 문제가 될 수 있다
a = torch.tensor([1.0, 2.0])              # float32
b = torch.from_numpy(np.array([3.0, 4.0]))  # float64
print(f"float32 + float64 = {(a + b).dtype}   (자동 승격)")
print()
print("권장: NumPy에서 가져올 때 자료형을 명시한다")
print("  torch.tensor(arr, dtype=torch.float32)")
print("  또는  torch.from_numpy(arr.astype(np.float32))")
print()
print("딥러닝에서 float32를 쓰는 이유: 메모리 절반, 속도는 빠르고,")
print("정밀도는 대부분의 경우 충분하기 때문이다 (이론편 22.5절).")

---

## 2. autograd로 이론편 10.4절 값 재현 ★

이 장에서 가장 중요한 부분이다.

11장에서 우리는 다음을 손으로 했다.

1. 미분식을 연쇄법칙으로 유도
2. 코드로 옮김
3. 수치 미분으로 검증

**PyTorch는 이 모두를 `loss.backward()` 한 줄로 처리한다.**
정말 같은 답이 나오는지 확인해 보자.

In [ ]:
import torch
import numpy as np

print("=" * 60)
print("PyTorch autograd — 이론편 10.4절 값 재현")
print("=" * 60)

# 이론편 10.4절과 완전히 같은 설정
x  = torch.tensor([1.0, 0.5])
W1 = torch.tensor([[0.2, 0.4],
                   [0.1, 0.3]], requires_grad=True)   # ← 미분 대상 표시
W2 = torch.tensor([0.6, 0.9], requires_grad=True)
y  = torch.tensor(1.0)

# --- 순전파: 11장과 똑같이 쓴다 ---
z1 = W1 @ x
a1 = torch.sigmoid(z1)
z2 = W2 @ a1
L  = (z2 - y) ** 2

print("[순전파]")
print(f"  z1 = {z1.detach().numpy().round(4)}        이론편: [0.40, 0.25]")
print(f"  a1 = {a1.detach().numpy().round(4)}    이론편: [0.5987, 0.5622]")
print(f"  z2 = {z2.item():.4f}                  이론편: 0.8652")
print(f"  L  = {L.item():.4f}                  이론편: 0.0182")

# --- 역전파: 한 줄이면 끝난다 ---
L.backward()

print()
print("[역전파]  L.backward()  한 줄")
print(f"  dL/dW2 = {W2.grad.numpy().round(4)}   이론편: [-0.1614, -0.1516]")
print(f"  dL/dW1 =")
print(f"{W1.grad.numpy().round(4)}")
print(f"  이론편  = [[-0.0389 -0.0194]")
print(f"            [-0.0597 -0.0299]]")

# --- 검증 ---
book_W1 = np.array([[-0.0389, -0.0194], [-0.0597, -0.0299]])
book_W2 = np.array([-0.1614, -0.1516])

print("-" * 60)
assert np.allclose(W1.grad.numpy(), book_W1, atol=5e-4)
assert np.allclose(W2.grad.numpy(), book_W2, atol=5e-4)
print("[OK] PyTorch가 계산한 값이 이론편 손계산과 일치")
print()
print("11장에서 40줄 넘게 짰던 역전파를 한 줄로 해냈다.")
print("하지만 그 안에서 무슨 일이 일어나는지 우리는 이미 안다.")

### `requires_grad`와 `detach`

| 표기 | 뜻 |
|---|---|
| `requires_grad=True` | 이 텐서에 대해 미분값을 구하겠다 |
| `.grad` | `backward()` 후 여기에 그래디언트가 담긴다 |
| `.detach()` | 계산 그래프에서 떼어낸다 (미분 추적 중단) |
| `.item()` | 값 하나짜리 텐서에서 파이썬 숫자를 꺼낸다 |

위에서 `z1.detach().numpy()`라고 쓴 이유는, 미분 대상인 텐서를 그냥 `.numpy()`하면
오류가 나기 때문이다. **출력하거나 기록할 때는 `detach()`를 붙인다.**

---

## 3. 계산 그래프 들여다보기

PyTorch가 어떻게 미분을 자동으로 할 수 있을까. **연산을 하는 동안 그래프를 만들어 두기 때문**이다.

이론편 5.3절의 연쇄법칙을 떠올려 보자. 각 단계의 국소적 미분을 곱해 나가는 것이었다.
PyTorch는 각 연산이 자기 미분법을 알고 있고, 어느 텐서에서 왔는지 기억한다.

In [ ]:
import torch

print("=" * 60)
print("계산 그래프 추적")
print("=" * 60)

a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)

c = a * b          # 곱셈
d = c + a          # 덧셈
e = d ** 2         # 제곱

print(f"a = {a.item()}, b = {b.item()}")
print(f"c = a * b   = {c.item()}")
print(f"d = c + a   = {d.item()}")
print(f"e = d ** 2  = {e.item()}")
print()

print("각 텐서가 기억하는 '자신을 만든 연산'")
for name, t in [("a", a), ("c", c), ("d", d), ("e", e)]:
    fn = t.grad_fn
    print(f"  {name}: {type(fn).__name__ if fn else '없음 (사용자가 만든 텐서)'}")
print()

e.backward()
print("backward() 후")
print(f"  de/da = {a.grad.item()}")
print(f"  de/db = {b.grad.item()}")
print()

# 손으로 확인
# e = d^2, d = c + a, c = a*b
# de/dd = 2d = 2*8 = 16
# dd/da = dc/da + 1 = b + 1 = 4      (a가 두 경로로 기여)
# de/da = 16 * 4 = 64
print("손으로 유도하면")
print("  e = d², d = c + a, c = a·b")
print("  de/dd = 2d = 2·8 = 16")
print("  dd/da = b + 1 = 4      ← a가 c를 통해서도, 직접도 기여")
print("  de/da = 16 × 4 = 64")
print(f"  → PyTorch 결과 {a.grad.item()} 와 일치")
assert a.grad.item() == 64.0
assert b.grad.item() == 32.0
print("[OK]")

### 그래디언트는 누적된다 — 자주 하는 실수

`backward()`를 두 번 호출하면 그래디언트가 **더해진다.** 덮어쓰는 것이 아니다.

이 성질은 여러 배치의 그래디언트를 모을 때 유용하지만(이론편 11.6절),
보통은 매 스텝마다 초기화해야 한다. 그래서 학습 루프에 `optimizer.zero_grad()`가 들어간다.

In [ ]:
import torch

print("=" * 55)
print("그래디언트 누적 확인")
print("=" * 55)

w = torch.tensor(1.0, requires_grad=True)

for i in range(1, 4):
    loss = w ** 2       # dL/dw = 2w = 2
    loss.backward()
    print(f"  {i}번째 backward 후 w.grad = {w.grad.item()}")

print()
print("→ 2, 4, 6 으로 계속 더해졌다")
print()

w.grad.zero_()
print(f"zero_() 호출 후: {w.grad.item()}")
loss = w ** 2
loss.backward()
print(f"다시 backward 후: {w.grad.item()}")
print()
print("학습 루프에서 optimizer.zero_grad()를 빠뜨리면")
print("이전 스텝의 그래디언트가 계속 쌓여 학습이 이상해진다.")

---

## 4. `nn.Module`로 신경망 만들기 — 이론편 10.2절

11장에서 만든 `SimpleNetwork`를 PyTorch 방식으로 다시 만든다.

`nn.Module`을 상속하면 다음이 자동으로 처리된다.

- 가중치 초기화 (이론편 11.4절)
- 파라미터 수집 (`.parameters()`)
- GPU 이동 (`.to(device)`)
- 학습/평가 모드 전환 (`.train()` / `.eval()`)

In [ ]:
import torch
import torch.nn as nn


class XORNet(nn.Module):
    """10~11장에서 만든 것과 같은 구조 (입력2 - 은닉4 - 출력1)"""

    def __init__(self, n_hidden=4):
        super().__init__()
        # ── nn.Linear 파라미터 ───────────────────────────────────────
        #   in_features   입력 차원.  **필수**
        #   out_features  출력 차원.  **필수**
        #   bias          편향 사용 여부.  기본값 True
        #                 BatchNorm 이 뒤에 오면 False 로 해도 된다
        #                 (BatchNorm 의 beta 가 같은 역할)
        #
        #   파라미터 수 = in_features x out_features + out_features
        #   예: Linear(784, 256) → 784x256 + 256 = 200,960 개
        # ──────────────────────────────────────────────────────────────
        self.fc1 = nn.Linear(2, n_hidden)     # 가중치와 편향을 함께 관리
        self.fc2 = nn.Linear(n_hidden, 1)

    def forward(self, x):
        h = torch.sigmoid(self.fc1(x))
        out = torch.sigmoid(self.fc2(h))
        return out


model = XORNet(n_hidden=4)

print("=" * 55)
print("모델 구조")
print("=" * 55)
print(model)
print()

print("파라미터")
total = 0
for name, p in model.named_parameters():
    print(f"  {name:<14}{str(tuple(p.shape)):<12}{p.numel():>4}개")
    total += p.numel()
print(f"  {'합계':<26}{total:>4}개")
print()
print("직접 세어 보면: (2×4 + 4) + (4×1 + 1) = 12 + 5 = 17")
assert total == 17

# 같은 구조를 더 짧게 쓰는 방법
compact = nn.Sequential(
    nn.Linear(2, 4), nn.Sigmoid(),
    nn.Linear(4, 1), nn.Sigmoid(),
)
print()
print("nn.Sequential로 쓰면 더 짧다:")
print(compact)

In [ ]:
import torch
import torch.nn as nn

# 10~11장과 같은 XOR 데이터
X = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

torch.manual_seed(42)     # 재현성 (이론편 11.6절)
model = XORNet(n_hidden=4)
# ── torch.optim.Adam 파라미터 ────────────────────────────────
#   params        학습할 파라미터. model.parameters() 로 전달
#                 일부만 학습하려면 리스트로 골라 준다
#   lr            학습률.  기본값 0.001
#                 예: 1e-3(기본) / 1e-4(파인튜닝) / 5e-5(LLM)
#                 가장 중요한 값 — 너무 크면 발산, 작으면 느림
#   betas         모멘텀 계수.  기본값 (0.9, 0.999)
#                 beta1: 그래디언트 이동평균 (방향)
#                 beta2: 그래디언트 제곱 이동평균 (보폭)
#                 거의 바꾸지 않는다
#   eps           0 나눗셈 방지.  기본값 1e-8
#                 혼합정밀도 학습에서는 1e-6 으로 올리기도
#   weight_decay  L2 정규화 강도.  기본값 0
#                 예: 1e-4, 1e-2 — Adam 에서는 AdamW 권장
#   amsgrad       AMSGrad 변형 사용.  기본값 False
# ──────────────────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
criterion = nn.MSELoss()

print("=" * 55)
print("XOR 학습 — PyTorch 표준 루프")
print("=" * 55)

losses = []
for epoch in range(2000):
    # --- 이 네 줄이 PyTorch 학습의 표준 형태다 ---
    optimizer.zero_grad()          # 1) 이전 그래디언트 지우기
    output = model(X)              # 2) 순전파
    loss = criterion(output, y)    # 3) 손실 계산
    loss.backward()                # 4) 역전파
    optimizer.step()               # 5) 가중치 갱신

    if epoch % 200 == 0:
        losses.append((epoch, loss.item()))
        print(f"  에폭 {epoch:5}: 손실 {loss.item():.6f}")

print()
with torch.no_grad():              # 평가 시에는 그래디언트 계산 불필요
    pred = model(X)

print(f"{'입력':<14}{'예측':<12}{'정답'}")
print("-" * 55)
for xi, pi, yi in zip(X, pred.ravel(), y.ravel()):
    print(f"{str(xi.numpy().astype(int)):<14}{pi.item():<12.4f}{int(yi.item())}")
print("-" * 55)
assert loss.item() < 0.01
print("[OK] 10장에서 NumPy로 푼 XOR을 PyTorch로도 풀었다")

### 학습 루프의 다섯 줄

방금 쓴 다섯 줄이 PyTorch의 표준 형태다. **앞으로 나오는 모든 학습 코드가 이 형태**다.

```python
optimizer.zero_grad()       # 이전 그래디언트 지우기
output = model(X)           # 순전파
loss = criterion(output, y) # 손실
loss.backward()             # 역전파
optimizer.step()            # 갱신
```

11장에서 손으로 했던 것과 대응해 보면 이렇다.

| PyTorch | 11장에서 한 일 |
|---|---|
| `model(X)` | `forward()` — 중간값 저장하며 계산 |
| `loss.backward()` | `backward()` — delta를 뒤에서 앞으로 |
| `optimizer.step()` | `W -= lr * grad` |
| `zero_grad()` | (필요 없었음 — 매번 새로 계산했으므로) |

`with torch.no_grad():`는 평가할 때 쓴다. 그래디언트를 만들지 않으므로
메모리를 아끼고 속도도 빨라진다.

---

## 5. 옵티마이저 비교 — 이론편 11.1절

이론편 11.1절에서 SGD·모멘텀·Adam을 다뤘다. PyTorch에서는 한 줄만 바꾸면 된다.

같은 문제에 세 가지를 적용해 차이를 본다.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

def train_with(optimizer_name, epochs=500, seed=42):
    torch.manual_seed(seed)
    model = XORNet(n_hidden=4)
    criterion = nn.MSELoss()

    opts = {
        # ── torch.optim.SGD 파라미터 ─────────────────────────────────
        #   lr           학습률.  **기본값 없음 (필수)**
        #                Adam 보다 크게 잡는다. 예: 0.01~0.1
        #   momentum     관성 계수.  기본값 0
        #                0.9 가 관행 — 지그재그를 줄이고 가속한다
        #   dampening    모멘텀 감쇠.  기본값 0
        #   weight_decay L2 정규화.  기본값 0. 예: 5e-4 (CNN 학습)
        #   nesterov     네스테로프 모멘텀.  기본값 False
        #                True 로 하면 조금 더 나은 수렴 (momentum 필요)
        # ──────────────────────────────────────────────────────────────
        "SGD":       torch.optim.SGD(model.parameters(), lr=0.5),
        "Momentum":  torch.optim.SGD(model.parameters(), lr=0.5, momentum=0.9),
        "Adam":      torch.optim.Adam(model.parameters(), lr=0.1),
    }
    optimizer = opts[optimizer_name]

    history = []
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    return history

print("=" * 55)
print("옵티마이저 비교 (이론편 11.1절)")
print("=" * 55)

results = {}
for name in ["SGD", "Momentum", "Adam"]:
    h = train_with(name)
    results[name] = h
    # 손실이 0.01 아래로 처음 내려간 시점
    reached = next((i for i, v in enumerate(h) if v < 0.01), None)
    when = f"{reached}에폭" if reached else "도달 못함"
    print(f"  {name:<12} 최종 손실 {h[-1]:.6f}   0.01 도달: {when}")

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = {"SGD": "#1E40AF", "Momentum": "#EA580C", "Adam": "#0D9488"}
for name, h in results.items():
    ax.plot(h, label=name, linewidth=2, color=colors[name])
ax.set_xlabel("에폭")
ax.set_ylabel("손실")
ax.set_yscale("log")
ax.set_title("옵티마이저별 수렴 속도")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print()
print("이론편 11.1절에서 다룬 대로, 모멘텀과 Adam이 SGD보다 빠르게 수렴한다.")
print("Adam은 파라미터마다 학습률을 조절하므로 초기 설정에 덜 민감하다.")

---

## 6. 실제 데이터로 학습 — 이론편 11.6절

지금까지 XOR은 데이터가 4개뿐이라 배치를 나눌 필요가 없었다.
데이터가 많을 때 쓰는 **`DataLoader`**를 익힌다.

이론편 11.6절에서 다룬 에폭·배치·셔플을 PyTorch가 어떻게 처리하는지 본다.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# 08장에서 쓴 반달 데이터
Xm, ym = make_moons(n_samples=1000, noise=0.2, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=0.3,
                                      random_state=42, stratify=ym)

sc = StandardScaler().fit(Xtr)          # 훈련 데이터로만 fit (07장 참조)
Xtr_t = torch.tensor(sc.transform(Xtr), dtype=torch.float32)
Xte_t = torch.tensor(sc.transform(Xte), dtype=torch.float32)
ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1)
yte_t = torch.tensor(yte, dtype=torch.float32).unsqueeze(1)

# DataLoader — 배치 나누기와 셔플을 자동으로 (이론편 11.6절)
train_ds = TensorDataset(Xtr_t, ytr_t)
# ── DataLoader 파라미터 ──────────────────────────────────────
#   dataset      Dataset 객체
#   batch_size   한 번에 처리할 표본 수.  기본값 1
#                예: 32, 64, 128 — 크면 빠르지만 메모리 많이 씀
#                GPU 메모리 부족 시 가장 먼저 줄이는 값
#   shuffle      매 에폭 섞을지.  기본값 False
#                **학습은 True, 검증·시험은 False**
#   num_workers  데이터 로딩 병렬 프로세스.  기본값 0
#                Windows 는 0 을 권장 (프로세스 생성 비용)
#                Linux 는 4~8 정도가 일반적
#   pin_memory   고정 메모리 사용.  기본값 False
#                GPU 학습 시 True 로 하면 전송이 빨라진다
#   drop_last    마지막 불완전 배치 버림.  기본값 False
#                BatchNorm 사용 시 True 권장 (배치 1 방지)
# ──────────────────────────────────────────────────────────────
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)

print("=" * 55)
print("DataLoader (이론편 11.6절)")
print("=" * 55)
print(f"훈련 데이터  : {len(train_ds)}개")
print(f"배치 크기    : 32")
print(f"1에폭 스텝 수 : {len(train_dl)}   ({len(train_ds)} / 32 올림)")
print()

# 첫 배치만 확인
first_x, first_y = next(iter(train_dl))
print(f"한 배치의 모양: X {tuple(first_x.shape)}, y {tuple(first_y.shape)}")
print()
print("shuffle=True 이므로 에폭마다 순서가 바뀐다 (이론편 11.6절의 무작위성)")

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),      # 이론편 10.3절 — ReLU 사용
    nn.Linear(16, 8), nn.ReLU(),
    nn.Linear(8, 1),
)
criterion = nn.BCEWithLogitsLoss()     # 이진 분류용 (이론편 6.5절 교차 엔트로피)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print("=" * 55)
print("학습 (에폭 30, 배치 32)")
print("=" * 55)

train_losses, test_losses = [], []

for epoch in range(30):
    # --- 훈련 ---
    model.train()
    epoch_loss = 0.0
    for xb, yb in train_dl:            # 배치 단위 반복 (이론편 11.6절)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb)
    train_losses.append(epoch_loss / len(train_ds))

    # --- 평가 ---
    model.eval()
    with torch.no_grad():
        test_losses.append(criterion(model(Xte_t), yte_t).item())

    if epoch % 5 == 0:
        print(f"  에폭 {epoch:3}: 훈련 {train_losses[-1]:.4f}  검증 {test_losses[-1]:.4f}")

# 정확도
model.eval()
with torch.no_grad():
    pred = (torch.sigmoid(model(Xte_t)) > 0.5).float()
    acc = (pred == yte_t).float().mean().item()
print()
print(f"시험 정확도: {acc:.4f}")

# 03장에서 배운 손실 곡선 그리기
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(train_losses, label="훈련 손실", linewidth=2)
ax.plot(test_losses, label="검증 손실", linewidth=2, linestyle="--")
ax.set_xlabel("에폭")
ax.set_ylabel("손실")
ax.set_title("학습 곡선 (이론편 11.6절)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("두 곡선이 함께 내려가고 격차가 작으면 정상이다 (03장 3절).")

### `model.train()`과 `model.eval()`

두 모드를 전환하는 이유는 **일부 층이 학습 때와 평가 때 다르게 동작하기 때문**이다.

| 층 | 학습 시 | 평가 시 |
|---|---|---|
| Dropout (이론편 11.3절) | 일부 뉴런을 끈다 | 전부 사용 |
| BatchNorm (이론편 11.3절) | 배치 통계 사용 | 누적 통계 사용 |

지금 모델에는 이런 층이 없어 차이가 없지만, **습관적으로 넣어 두는 것**이 좋다.
나중에 Dropout을 추가했을 때 이것을 빠뜨리면 평가 결과가 이상해진다.

---

## 7. GPU 사용

GPU가 있다면 텐서와 모델을 GPU로 옮겨 계산할 수 있다.
없어도 이 장의 모든 내용은 CPU로 동작한다.

In [ ]:
import torch
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 55)
print(f"사용 장치: {device}")
print("=" * 55)

if device.type == "cuda":
    print(f"GPU 이름 : {torch.cuda.get_device_name(0)}")
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"메모리   : {total:.1f} GB")
    print()

    # 속도 비교
    size = 3000
    a = torch.randn(size, size)
    b = torch.randn(size, size)

    t0 = time.time(); _ = a @ b; cpu_t = time.time() - t0

    a_g, b_g = a.to(device), b.to(device)
    _ = a_g @ b_g                     # 워밍업
    torch.cuda.synchronize()          # 01장에서 다룬 동기화
    t0 = time.time(); _ = a_g @ b_g
    torch.cuda.synchronize(); gpu_t = time.time() - t0

    print(f"{size}x{size} 행렬 곱")
    print(f"  CPU : {cpu_t*1000:8.1f} ms")
    print(f"  GPU : {gpu_t*1000:8.1f} ms  ({cpu_t/gpu_t:.1f}배)")
else:
    print("GPU가 없어 CPU로 실행합니다.")
    print("이 책의 대부분(1~23장)은 CPU로 충분합니다.")

print()
print("모델과 데이터를 GPU로 옮기는 방법")
print("  model = model.to(device)")
print("  xb, yb = xb.to(device), yb.to(device)")
print()
print("주의: 모델과 데이터가 **같은 장치**에 있어야 한다.")
print("  하나만 옮기면 'Expected all tensors to be on the same device' 오류가 난다.")

---

## 8. 정리

### 확인한 값

| 대조 | 내용 | 결과 |
|---|---|---|
| 이론편 10.4절 | 순전파 4개 + 역전파 그래디언트 6개 | autograd 결과와 일치 ✓ |
| 10장 | XOR 학습 | PyTorch로도 해결 ✓ |
| 이론편 11.1절 | Adam이 SGD보다 빠름 | 곡선으로 확인 ✓ |

### PyTorch 학습의 표준 형태

```python
for epoch in range(n_epochs):
    model.train()
    for xb, yb in train_dl:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val), y_val)
```

**앞으로 나오는 모든 학습 코드가 이 형태다.** 12번(CNN), 13번(RNN), 이후 모두 같다.

### 기억할 것

| 항목 | 요점 |
|---|---|
| `requires_grad=True` | 미분 대상 표시 |
| `.detach()` | 출력·기록 시 그래프에서 분리 |
| `zero_grad()` | **빠뜨리면 그래디언트가 누적됨** |
| `no_grad()` | 평가 시 메모리·속도 절약 |
| `train()` / `eval()` | Dropout·BatchNorm 동작 전환 |
| `DataLoader` | 배치 나누기 + 셔플 자동 처리 |
| 자료형 | PyTorch 기본은 float32 |
| GPU | 모델과 데이터가 같은 장치에 있어야 함 |

### Part 2를 마치며

07~12장에서 다음을 했다.

| 장 | 한 일 |
|---|---|
| 06 | scikit-learn 지도학습, 정확도의 함정 |
| 07 | 비지도학습, K-Means 직접 구현 |
| 08 | 퍼셉트론의 한계와 MLP |
| 09 | **역전파 직접 구현 — 이론편 17개 값 검증** |
| 10 | PyTorch — 같은 것을 자동으로 |

**11장에서 직접 만들고 12장에서 자동화된 도구를 쓰는 순서**가 이 책의 방식이다.
원리를 알고 도구를 쓰는 것과, 도구만 아는 것은 다르다.

### 다음 장

**13. 최적화 알고리즘 — SGD에서 Adam까지** — 3부가 시작된다. 이론편 16장의 합성곱을 직접 구현해
**이론편 12.2절에서 손으로 구한 값 (0, −210)**을 확인하고, 이미지 분류 모델을 만든다.